In [ ]:


# Analysis code
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import shap
import sklearn
import warnings
warnings.filterwarnings("ignore")



# Use root_mean_squared_error if available (scikit-learn >= 1.4)
try:
    from sklearn.metrics import root_mean_squared_error
    use_rmse_directly = True
except ImportError:
    use_rmse_directly = False

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette('deep')

# Load dataset
df = pd.read_csv('../data/MachineLearningRating_v3.txt', sep='|')

# Data Preparation: Handle missing values
numeric_cols = ['CustomValueEstimate', 'SumInsured', 'cubiccapacity', 'kilowatts', 'RegistrationYear', 'NumberOfVehiclesInFleet', 'Cylinders', 'NumberOfDoors', 'TotalPremium']
categorical_cols = ['Province', 'VehicleType', 'MaritalStatus', 'Gender']

# Impute missing values for numeric columns
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col].fillna(df[col].median(), inplace=True)

# Impute missing values for categorical columns
for col in categorical_cols:
    if col in df.columns:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Feature Engineering
if 'RegistrationYear' in df.columns:
    df['VehicleAge'] = 2025 - df['RegistrationYear']
    df['VehicleAge'].fillna(df['VehicleAge'].median(), inplace=True)
else:
    print("Warning: RegistrationYear not found. Setting VehicleAge to 0.")
    df['VehicleAge'] = 0

if 'TotalClaims' in df.columns:
    df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
else:
    print("Warning: TotalClaims not found. Setting HasClaim to 0.")
    df['HasClaim'] = 0

if 'TotalPremium' in df.columns and 'TotalClaims' in df.columns:
    df['Margin'] = df['TotalPremium'] - df['TotalClaims']
else:
    print("Warning: TotalPremium or TotalClaims not found. Setting Margin to 0.")
    df['Margin'] = 0

if 'CustomValueEstimate' in df.columns and 'TotalPremium' in df.columns:
    df['ValueToCoverRatio'] = df['CustomValueEstimate'] / (df['TotalPremium'] + 1)
    df['ValueToCoverRatio'].fillna(df['ValueToCoverRatio'].median(), inplace=True)
else:
    print("Warning: CustomValueEstimate or TotalPremium not found. Using SumInsured for ValueToCoverRatio.")
    if 'SumInsured' in df.columns and 'TotalPremium' in df.columns:
        df['ValueToCoverRatio'] = df['SumInsured'] / (df['TotalPremium'] + 1)
        df['ValueToCoverRatio'].fillna(df['ValueToCoverRatio'].median(), inplace=True)
    else:
        df['ValueToCoverRatio'] = 0

# Categorical Encoding
categorical = ['Province', 'VehicleType']
if 'MaritalStatus' in df.columns:
    categorical.append('MaritalStatus')
elif 'Gender' in df.columns:
    categorical.append('Gender')
elif 'Title' in df.columns:
    print("Warning: MaritalStatus and Gender not found. Using Title as fallback.")
    categorical.append('Title')

# Filter categorical columns to those available
categorical = [col for col in categorical if col in df.columns]
if categorical:
    df = pd.get_dummies(df, columns=categorical, drop_first=True)
else:
    print("Warning: No valid categorical columns for encoding.")

# Select common features
feature_cols = [col for col in df.columns if col not in ['TotalClaims', 'CalculatedPremiumPerTerm',
                                                         'TotalPremium', 'HasClaim', 'Margin']]
# Exclude non-numeric or irrelevant columns
exclude_cols = ['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'Citizenship', 'LegalType',
                'Language', 'Bank', 'AccountType', 'Country', 'MainCrestaZone', 'SubCrestaZone',
                'ItemType', 'mmcode', 'make', 'Model', 'bodytype', 'VehicleIntroDate',
                'AlarmImmobiliser', 'TrackingDevice', 'NewVehicle', 'WrittenOff', 'Rebuilt',
                'Converted', 'CrossBorder', 'StatutoryClass', 'StatutoryRiskType']
feature_cols = [col for col in feature_cols if col not in exclude_cols and df[col].dtype in ['float64', 'int64', 'uint8', 'bool']]
print(f"Using features: {feature_cols}")

# Identify numeric and dummy columns for preprocessing
raw_numeric_cols = ['CustomValueEstimate', 'SumInsured', 'cubiccapacity', 'kilowatts', 'VehicleAge', 'NumberOfVehiclesInFleet', 'Cylinders', 'NumberOfDoors', 'ValueToCoverRatio']
raw_numeric_cols = [col for col in raw_numeric_cols if col in feature_cols]
dummy_cols = [col for col in feature_cols if col not in raw_numeric_cols]
print(f"Raw numeric columns: {raw_numeric_cols}")
print(f"Dummy columns: {dummy_cols}")

# Check for NaNs in features
print("NaN counts in feature columns:")
print(df[feature_cols].isnull().sum())

# Preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Apply preprocessing only to raw numeric columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, raw_numeric_cols),
        ('passthrough', 'passthrough', dummy_cols)
    ])

# 1. Claim Severity Model (Only where claims > 0)
df_severity = df[df['TotalClaims'] > 0].copy()
if df_severity.empty or not feature_cols:
    print("No data with TotalClaims > 0 or no valid features for severity prediction. Skipping.")
    severity_results = {}
else:
    X_s, y_s = df_severity[feature_cols], df_severity['TotalClaims']
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_s, y_s, test_size=0.2, random_state=42)

    # Train models
    models = {
        'LinearRegression': Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())]),
        'RandomForest': Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(random_state=42))]),
        'XGBoost': Pipeline([('preprocessor', preprocessor), ('model', XGBRegressor(random_state=42))])
    }

    print("\n---- Claim Severity Model Evaluation ----")
    severity_results = {}
    for name, pipeline in models.items():
        pipeline.fit(X_train_s, y_train_s)
        y_pred = pipeline.predict(X_test_s)
        if use_rmse_directly:
            rmse = root_mean_squared_error(y_test_s, y_pred)
        else:
            rmse = np.sqrt(mean_squared_error(y_test_s, y_pred))
        r2 = r2_score(y_test_s, y_pred)
        severity_results[name] = {'RMSE': rmse, 'R2': r2}
        print(f"{name} | RMSE: {rmse:.2f} | R2: {r2:.3f}")

    # Save best model for SHAP
    best_model = models['XGBoost']

# 2. Premium Prediction Model
if 'CalculatedPremiumPerTerm' in df.columns and feature_cols:
    X_p, y_p = df[feature_cols], df['CalculatedPremiumPerTerm']
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

    premium_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(random_state=42))
    ])
    premium_pipeline.fit(X_train_p, y_train_p)
    y_pred_premium = premium_pipeline.predict(X_test_p)

    if use_rmse_directly:
        rmse_premium = root_mean_squared_error(y_test_p, y_pred_premium)
    else:
        rmse_premium = np.sqrt(mean_squared_error(y_test_p, y_pred_premium))
    r2_premium = r2_score(y_test_p, y_pred_premium)

    print("\n---- Premium Prediction ----")
    print(f"RMSE: {rmse_premium:.2f} | R2: {r2_premium:.3f}")
else:
    print("Warning: CalculatedPremiumPerTerm or valid features not found. Skipping premium prediction.")
    rmse_premium = r2_premium = None

# 3. Claim Occurrence Prediction (Classification)
if feature_cols:
    X_c, y_c = df[feature_cols], df['HasClaim'].astype(int)
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

    classifier_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(random_state=42))
    ])
    classifier_pipeline.fit(X_train_c, y_train_c)
    y_pred_c = classifier_pipeline.predict(X_test_c)

    print("\n---- Claim Occurrence Classification ----")
    print(classification_report(y_test_c, y_pred_c, zero_division=0))
else:
    print("Warning: No valid features for claim occurrence prediction. Skipping.")

# 4. SHAP Analysis on Claim Severity Model
if severity_results and feature_cols:
    print("\n---- SHAP Feature Importance (Claim Severity) ----")
    # Extract the model from the pipeline
    best_model_pipeline = best_model
    best_model = best_model_pipeline.named_steps['model']
    X_test_s_transformed = best_model_pipeline.named_steps['preprocessor'].transform(X_test_s)

    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer(X_test_s_transformed)

    # Get feature names after preprocessing
    feature_names = preprocessor.get_feature_names_out()
    feature_names = [name.replace('num__', '').replace('passthrough__', '') for name in feature_names]
    print(f"Preprocessed feature names: {feature_names}")
    print(f"Shape of X_test_s: {X_test_s.shape}")
    print(f"Shape of X_test_s_transformed: {X_test_s_transformed.shape}")
    print(f"Shape of shap_values: {shap_values.values.shape}")
    print(f"Number of feature names: {len(feature_names)}")

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test_s_transformed, feature_names=feature_names, max_display=10, show=False)
    plt.title('SHAP Feature Importance for Claim Severity (XGBoost)')
    plt.show()

    # Top 5 Features
    shap_importance = pd.DataFrame({
        'Feature': feature_names,
        'Importance': np.abs(shap_values.values).mean(axis=0)
    })
    top_features = shap_importance.sort_values(by='Importance', ascending=False).head(5)
    print("\n=== Top 5 Features for Claim Severity (XGBoost) ===")
    print(top_features)

    # Business Interpretation
    print("\n=== Business Interpretation of Top Features ===")
    for _, row in top_features.iterrows():
        feature = row['Feature']
        importance = row['Importance']
        if 'VehicleAge' in feature:
            print(f"VehicleAge: Older vehicles increase claim severity by {importance:.2f}. Consider higher premiums for older vehicles.")
        elif 'CustomValueEstimate' in feature:
            print(f"CustomValueEstimate: Higher vehicle values increase claim severity by {importance:.2f}. Premiums should scale with vehicle value.")
        elif 'SumInsured' in feature:
            print(f"SumInsured: Higher insured values increase claim severity by {importance:.2f}. Premiums should scale with insured value.")
        elif 'Province' in feature:
            print(f"Province: Certain provinces increase claim severity by {importance:.2f}. Adjust premiums regionally.")
        elif 'VehicleType' in feature:
            print(f"VehicleType: Specific vehicle types impact claim severity by {importance:.2f}. Tailor premiums by vehicle type.")
        elif 'MaritalStatus' in feature or 'Gender' in feature or 'Title' in feature:
            print(f"{feature}: Demographic factors impact claim severity by {importance:.2f}. Consider demographic-based pricing adjustments.")

